# Final Walk-Forward VaR Results Analysis

## Objective

Reproduce and inspect the final Day-21 walk-forward evaluation for the
equal-weight HPG/FPT/MWG portfolio across Historical Simulation, EWMA, and
Gradient Boosting Quantile Regression.

This notebook is an analysis and reconciliation layer. It does not retune
parameters, add features, perform model selection, or declare an overall
winner.

The common evaluation universe contains 398 one-day-ahead target dates from
2024-12-18 through 2026-07-28 at the 5% return quantile.

## Method

The notebook reuses the tested Day-22 analysis modules rather than
reimplementing their quantitative logic.

It performs three checks:

1. recompute the aggregate metric comparison from canonical predictions;
2. rebuild the 398-date paired diagnostic table;
3. rebuild the pairwise magnitude/concentration summary.

Each live result is reconciled against its saved Day-22 CSV artifact.

The sign convention for pairwise loss differences is:

`left method Pinball Loss - right method Pinball Loss`.

Therefore, a negative mean difference favors the left-hand method.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    current = Path.cwd().resolve()

    candidates = [
        current,
        current.parent,
    ]

    for candidate in candidates:
        if (
            (candidate / "results" / "final_predictions.csv").is_file()
            and (candidate / "scripts").is_dir()
        ):
            return candidate

    raise RuntimeError(
        "Unable to locate repository root from notebook execution directory."
    )


REPO_ROOT = find_repo_root()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository root:", REPO_ROOT)
print("Notebook setup: PASS")

Repository root: C:\Users\Admin\Downloads\portfolio-var-risk-system
Notebook setup: PASS


In [2]:
from scripts.analyze_final_metrics_a import build_comparison

from scripts.analyze_final_pairwise_a import (
    build_pairwise_diagnostics,
    load_predictions,
)

from scripts.analyze_pairwise_concentration_a import build_summary


comparison_live = build_comparison()

predictions = load_predictions()

paired_live = build_pairwise_diagnostics(
    predictions
)

summary_live = build_summary(
    paired_live
)

print("Metric comparison rows :", len(comparison_live))
print("Paired target rows     :", len(paired_live))
print("Pairwise summary rows  :", len(summary_live))

assert len(comparison_live) == 3
assert len(paired_live) == 398
assert len(summary_live) == 3

print("Live analysis reconstruction: PASS")

Metric comparison rows : 3
Paired target rows     : 398
Pairwise summary rows  : 3
Live analysis reconstruction: PASS


## Validation

The following checks compare the live calculations above against the persisted
Day-22 analytical outputs.

The notebook therefore acts as a reproducibility consumer of the scripts,
rather than as a second independent implementation of the model or metric
formulae.

In [3]:
import numpy as np

metric_saved = pd.read_csv(
    REPO_ROOT / "results" / "final_metric_comparison.csv",
    parse_dates=["test_start", "test_end"],
)

paired_saved = pd.read_csv(
    REPO_ROOT / "results" / "final_pairwise_diagnostics.csv",
    parse_dates=["target_date"],
)

summary_saved = pd.read_csv(
    REPO_ROOT / "results" / "final_pairwise_summary.csv",
)


comparison_check = comparison_live.copy()

for column in ["test_start", "test_end"]:
    comparison_check[column] = pd.to_datetime(
        comparison_check[column]
    )

paired_check = paired_live.copy()

paired_check["target_date"] = pd.to_datetime(
    paired_check["target_date"]
)


pd.testing.assert_frame_equal(
    comparison_check.reset_index(drop=True),
    metric_saved.reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=0.0,
    atol=1e-15,
)

pd.testing.assert_frame_equal(
    paired_check.reset_index(drop=True),
    paired_saved.reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=0.0,
    atol=1e-15,
)

summary_live_check = summary_live.reset_index(
    drop=True
)

summary_saved_check = summary_saved.reset_index(
    drop=True
)

summary_exact_columns = [
    "comparison",
    "left_method",
    "right_method",
    "observation_count",
    "left_better_count",
    "right_better_count",
    "tie_count",
    "improvement_dates_to_50pct",
    "improvement_dates_to_80pct",
    "largest_left_improvement_date",
    "largest_left_deterioration_date",
]

pd.testing.assert_frame_equal(
    summary_live_check[
        summary_exact_columns
    ],
    summary_saved_check[
        summary_exact_columns
    ],
    check_dtype=False,
    check_exact=True,
)

summary_numeric_columns = [
    column
    for column in summary_live_check.columns
    if column not in summary_exact_columns
]

for column in summary_numeric_columns:
    live_values = pd.to_numeric(
        summary_live_check[column],
        errors="raise",
    ).to_numpy(dtype="float64")

    saved_values = pd.to_numeric(
        summary_saved_check[column],
        errors="raise",
    ).to_numpy(dtype="float64")

    if not np.allclose(
        live_values,
        saved_values,
        rtol=0.0,
        atol=1e-12,
    ):
        maximum_difference = float(
            np.max(
                np.abs(
                    live_values
                    - saved_values
                )
            )
        )

        raise AssertionError(
            f"{column}: summary reconciliation failed; "
            f"max abs difference={maximum_difference:.18e}"
        )

assert comparison_live["method"].tolist() == [
    "historical_simulation",
    "ewma",
    "gradient_boosting",
]

assert summary_live["comparison"].tolist() == [
    "gb_vs_historical",
    "gb_vs_ewma",
    "ewma_vs_historical",
]

print("Metric CSV reconciliation   : PASS")
print("Paired CSV reconciliation   : PASS")
print("Summary CSV reconciliation  : PASS")
print("NOTEBOOK_VALIDATION_PASS")

Metric CSV reconciliation   : PASS
Paired CSV reconciliation   : PASS
Summary CSV reconciliation  : PASS
NOTEBOOK_VALIDATION_PASS


## Result

The tables below expose the final aggregate comparison and the paired
loss-difference evidence used in the Day-22 results chapter.

The tables are descriptive evaluation outputs. They are not a new tuning
stage.

In [4]:
metric_display = comparison_live[
    [
        "method",
        "forecast_count",
        "violation_count",
        "violation_rate",
        "calibration_distance",
        "pinball_loss",
        "average_var",
    ]
].copy()

metric_display["violation_rate"] = (
    metric_display["violation_rate"] * 100.0
)

metric_display["calibration_distance"] = (
    metric_display["calibration_distance"] * 100.0
)

metric_display["average_var"] = (
    metric_display["average_var"] * 100.0
)

metric_display = metric_display.rename(
    columns={
        "violation_rate": "violation_rate_pct",
        "calibration_distance": "calibration_distance_pct",
        "average_var": "average_var_pct",
    }
)

display(metric_display.round(6))

,method,forecast_count,violation_count,violation_rate_pct,calibration_distance_pct,pinball_loss,average_var_pct
0,historical_simulation,398,27,6.783920,1.783920,0.001896,2.133019
1,ewma,398,22,5.527638,0.527638,0.001968,2.508404
2,gradient_boosting,398,24,6.030151,1.030151,0.001758,2.334906


In [5]:
pairwise_display = summary_live[
    [
        "comparison",
        "left_better_count",
        "right_better_count",
        "tie_count",
        "mean_delta",
        "median_delta",
        "top_5_improvement_share",
        "top_20_improvement_share",
        "improvement_dates_to_50pct",
        "improvement_dates_to_80pct",
    ]
].copy()

display(pairwise_display.round(6))

,comparison,left_better_count,right_better_count,tie_count,mean_delta,median_delta,top_5_improvement_share,top_20_improvement_share,improvement_dates_to_50pct,improvement_dates_to_80pct
0,gb_vs_historical,151,247,0,-0.000138,0.000056,0.437029,0.865487,7,16
1,gb_vs_ewma,231,167,0,-0.000210,-0.000053,0.297168,0.572150,13,63
2,ewma_vs_historical,131,267,0,0.000072,0.000078,0.662517,0.909384,3,8


## Interpretation

Interpretation remains criterion-specific and separates frequency from
magnitude.

The notebook derives the final statements below directly from the live
reconciled tables. Exception timing and volatility-regime interpretation are
outside Member A's scope and are intentionally not inferred here.

In [6]:
metric_lookup = comparison_live.set_index(
    "method"
)

pair_lookup = summary_live.set_index(
    "comparison"
)

historical = metric_lookup.loc[
    "historical_simulation"
]

ewma = metric_lookup.loc[
    "ewma"
]

gb = metric_lookup.loc[
    "gradient_boosting"
]

gb_hist = pair_lookup.loc[
    "gb_vs_historical"
]

gb_ewma = pair_lookup.loc[
    "gb_vs_ewma"
]


interpretation = f"""
### Criterion-specific interpretation

- **Calibration:** EWMA is closest to the nominal 5% violation rate
  ({100.0 * ewma['violation_rate']:.2f}% observed).
- **Quantile loss:** Gradient Boosting G04 has the lowest aggregate Pinball
  Loss ({gb['pinball_loss']:.12f}).
- **Average VaR:** Historical Simulation has the lowest Average VaR
  ({100.0 * historical['average_var']:.2f}%). Lower Average VaR is not
  automatically better.

### Paired evidence

Against Historical Simulation, Gradient Boosting has lower loss on
{int(gb_hist['left_better_count'])} dates and higher loss on
{int(gb_hist['right_better_count'])} dates, yet its mean paired difference is
{gb_hist['mean_delta']:.12f}. Its aggregate advantage is therefore
magnitude-driven rather than frequency-driven. Half of its total positive
improvement magnitude is reached in only
{int(gb_hist['improvement_dates_to_50pct'])} dates and 80% in
{int(gb_hist['improvement_dates_to_80pct'])} dates.

Against EWMA, Gradient Boosting has lower loss on
{int(gb_ewma['left_better_count'])} dates versus
{int(gb_ewma['right_better_count'])} dates favoring EWMA, with mean paired
difference {gb_ewma['mean_delta']:.12f}. This is a broader paired advantage
than the Gradient-Boosting-versus-Historical result.

### Model-selection status

**No overall model winner is declared.**

The evaluation period was reserved from parameter selection but is not
described as a pristine untouched test set. G04 remains an A-side validation
candidate rather than a confirmed final G01-G07 tuning winner.
"""

display(
    Markdown(interpretation)
)

print("NOTEBOOK_INTERPRETATION_PASS")


### Criterion-specific interpretation

- **Calibration:** EWMA is closest to the nominal 5% violation rate
  (5.53% observed).
- **Quantile loss:** Gradient Boosting G04 has the lowest aggregate Pinball
  Loss (0.001758130951).
- **Average VaR:** Historical Simulation has the lowest Average VaR
  (2.13%). Lower Average VaR is not
  automatically better.

### Paired evidence

Against Historical Simulation, Gradient Boosting has lower loss on
151 dates and higher loss on
247 dates, yet its mean paired difference is
-0.000138333282. Its aggregate advantage is therefore
magnitude-driven rather than frequency-driven. Half of its total positive
improvement magnitude is reached in only
7 dates and 80% in
16 dates.

Against EWMA, Gradient Boosting has lower loss on
231 dates versus
167 dates favoring EWMA, with mean paired
difference -0.000210190302. This is a broader paired advantage
than the Gradient-Boosting-versus-Historical result.

### Model-selection status

**No overall model winner is declared.**

The evaluation period was reserved from parameter selection but is not
described as a pristine untouched test set. G04 remains an A-side validation
candidate rather than a confirmed final G01-G07 tuning winner.


NOTEBOOK_INTERPRETATION_PASS
